# 資料分析

本章直接連 `cycle-purchase.db` 統計，**不從任何中介報表取數**。

```{admonition} 這一頁怎麼跑出真實數字
:class: important

預設會依序尋找資料庫：

1. 環境變數 `CP_DB`（自己指定路徑最保險）
2. `C:/portal_data/cycle-purchase.db`（正式／測試區的實際位置）
3. 專案 `backend/cycle-purchase.db`（開發預設）

**找不到就自動改用示範資料**，並在每張圖上打上「示範資料」浮水印。
浮水印出現＝那些數字不是你的資料，不要拿去回報。

在自己的機器上要看真實數字：

```bash
cd docs/jupyterbook/cycle-purchase
set CP_DB=C:\portal_data\cycle-purchase.db
jupyter-book build .
```
```

In [ ]:
import os, sqlite3, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 40)

# ── 中文字型 ────────────────────────────────────────────────────────────────
for name in ["Noto Sans CJK TC", "Noto Sans CJK SC", "Microsoft JhengHei", "PingFang TC"]:
    if any(name in f.name for f in font_manager.fontManager.ttflist):
        matplotlib.rcParams["font.sans-serif"] = [name]
        break
matplotlib.rcParams["axes.unicode_minus"] = False

# ── 找資料庫 ────────────────────────────────────────────────────────────────
CANDIDATES = [
    os.environ.get("CP_DB"),
    r"C:/portal_data/cycle-purchase.db",
    "../../../backend/cycle-purchase.db",
    "../../../cycle-purchase.db",
]
DB_PATH = next((p for p in CANDIDATES if p and Path(p).exists()), None)
IS_DEMO = DB_PATH is None

# 品牌色（CLAUDE.md 受保護色）
NAVY, SKY = "#1B3A5C", "#4BA8E8"
PALETTE = [NAVY, SKY, "#667eea", "#764ba2", "#e67e22", "#c0392b", "#2e8b57", "#7f8c8d"]

print(f"資料來源：{'（找不到資料庫，使用示範資料）' if IS_DEMO else DB_PATH}")

In [ ]:
def watermark(ax):
    """示範資料時在圖上打浮水印，避免被誤當成真實數字。"""
    if IS_DEMO:
        ax.text(0.5, 0.5, "示範資料", transform=ax.transAxes,
                fontsize=34, color="#c0392b", alpha=0.16,
                ha="center", va="center", rotation=22, zorder=10)

def q(sql, params=()):
    """對 cycle-purchase.db 查詢；沒有資料庫時回空 DataFrame。"""
    if IS_DEMO:
        return pd.DataFrame()
    with sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True) as conn:
        return pd.read_sql_query(sql, conn, params=params)

# ── 示範資料產生器（僅在找不到資料庫時使用；固定亂數種子，每次建置結果一致）──
rng = np.random.default_rng(20260820)
PERIODS = [f"2026-{m:02d}" for m in range(1, 9)]
DEPTS = ["工務部", "清潔部", "行政部", "營業部", "客務部"]
COMPANIES = ["春大直", "日曜天地"]

def demo_requests():
    rows = []
    for p in PERIODS:
        for d in DEPTS:
            for co in COMPANIES:
                rows.append({
                    "period_label": p, "department": d, "company": co,
                    "total_amount": float(rng.integers(8_000, 120_000)),
                    "is_closed": 1 if p < "2026-08" else int(rng.integers(0, 2)),
                    "is_summarized": 1 if p < "2026-07" else int(rng.integers(0, 2)),
                    "close_batch_no": ("CPAUTO-" + p.replace("-", "")) if rng.random() < 0.35
                                       else ("CPCLOSE-" + p.replace("-", "") + "-001"),
                })
    return pd.DataFrame(rows)

print("工具函式就緒。")

## 資料量體

先看每張表有多少筆——這是判斷「某個統計是不是根本沒資料」最快的方法。

In [ ]:
TABLES = [
    ("cycle_purchase_items", "料號主檔"),
    ("cycle_purchase_item_mappings", "料號對照表"),
    ("cycle_purchase_vendors", "供應商"),
    ("cycle_purchase_departments", "部門"),
    ("cycle_purchase_cost_centers", "成本中心"),
    ("cycle_purchase_account_codes", "會計科目"),
    ("cycle_purchase_categories", "類別"),
    ("cycle_purchase_cycles", "週期設定"),
    ("cycle_purchase_requests", "請購單"),
    ("cycle_purchase_request_items", "請購明細"),
    ("cycle_purchase_summary", "彙整列"),
    ("cycle_purchase_pos", "採購單"),
    ("cycle_purchase_po_items", "採購明細"),
    ("cycle_purchase_receiving", "驗收單"),
    ("cycle_purchase_receiving_items", "驗收明細"),
    ("cycle_purchase_payments", "請款單"),
    ("cycle_purchase_payment_allocations", "費用分攤"),
    ("cycle_purchase_audit_logs", "稽核紀錄"),
]

if IS_DEMO:
    counts = pd.DataFrame({
        "資料表": [t for t, _ in TABLES],
        "中文名": [n for _, n in TABLES],
        "筆數": ["—"] * len(TABLES),
    })
else:
    recs = []
    for t, n in TABLES:
        try:
            v = q(f"SELECT COUNT(*) AS c FROM {t}")["c"].iloc[0]
        except Exception:
            v = "表不存在"
        recs.append({"資料表": t, "中文名": n, "筆數": v})
    counts = pd.DataFrame(recs)

counts

## 請購單：期別 × 金額

請購單的 `total_amount` 是明細加總、由 `_recompute_total()` 系統維護，
所以可以直接加總，不需要回頭掃明細。

In [ ]:
if IS_DEMO:
    req = demo_requests()
else:
    req = q("""
        SELECT r.period_label,
               r.company,
               d.dept_name AS department,
               r.total_amount,
               r.is_closed,
               r.is_summarized,
               COALESCE(r.close_batch_no, '') AS close_batch_no
        FROM cycle_purchase_requests r
        LEFT JOIN cycle_purchase_departments d ON d.id = r.department_id
    """)

if len(req) == 0:
    print("目前沒有請購單資料。")
else:
    req["total_amount"] = pd.to_numeric(req["total_amount"], errors="coerce").fillna(0)
    by_period = (req.groupby(["period_label", "company"])["total_amount"]
                    .sum().unstack(fill_value=0).sort_index())

    fig, ax = plt.subplots(figsize=(9, 4.2))
    by_period.plot(kind="bar", ax=ax, color=PALETTE[:by_period.shape[1]], width=0.75)
    ax.set_title("請購金額（依期別／公司別）", fontsize=13, color=NAVY, pad=12)
    ax.set_xlabel(""); ax.set_ylabel("金額")
    ax.legend(title="公司別", frameon=False)
    ax.grid(axis="y", alpha=0.25)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    watermark(ax)
    plt.tight_layout(); plt.show()

    display(by_period.assign(合計=by_period.sum(axis=1)).round(0))

## 請購單狀態：開放中／人工關閉／系統自動關閉

人工關與系統關**不是欄位**，是靠 `close_batch_no` 前綴推出來的衍生值
（`CPAUTO-` ＝ 系統關）。這裡重現 `close_kind_of()` 的判斷。

In [ ]:
if len(req) == 0:
    print("沒有請購單資料。")
else:
    def close_kind(row):
        if not row["is_closed"]:
            return "開放中"
        return "系統自動關閉" if str(row["close_batch_no"]).startswith("CPAUTO-") else "人工關閉"

    req["關閉方式"] = req.apply(close_kind, axis=1)
    dist = req["關閉方式"].value_counts().reindex(
        ["開放中", "人工關閉", "系統自動關閉"]).fillna(0)

    fig, ax = plt.subplots(figsize=(5.6, 5.6))
    ax.pie(dist.values, labels=dist.index, autopct="%1.1f%%",
           colors=[SKY, NAVY, "#7f8c8d"], startangle=90,
           wedgeprops={"width": 0.42, "edgecolor": "white"},
           textprops={"fontsize": 11})
    ax.set_title("請購單關閉方式分佈", fontsize=13, color=NAVY, pad=14)
    watermark(ax)
    plt.tight_layout(); plt.show()

    summ = req["is_summarized"].astype(int).value_counts()
    print(f"已彙整：{summ.get(1, 0)} 張　／　未彙整：{summ.get(0, 0)} 張")

## 部門別金額

彙整的粒度是「公司＋料號＋部門」，這也是費用分攤能拆到部門的原因。
這裡用請購單層級看部門負擔。

In [ ]:
if len(req) == 0:
    print("沒有請購單資料。")
else:
    by_dept = (req.groupby("department")["total_amount"].sum()
                  .sort_values(ascending=True).tail(12))

    fig, ax = plt.subplots(figsize=(8.4, max(3.2, 0.42 * len(by_dept))))
    ax.barh(by_dept.index, by_dept.values, color=NAVY, height=0.62)
    ax.set_title("請購金額（依部門，前 12 名）", fontsize=13, color=NAVY, pad=12)
    ax.set_xlabel("金額"); ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    watermark(ax)
    plt.tight_layout(); plt.show()

## 流程漏斗：請購 → 彙整 → 採購 → 驗收 → 請款

看整條流程哪一段卡住。分母不同（張數 vs 列數）**不能互相相除算轉換率**，
這裡只看絕對量。

In [ ]:
if IS_DEMO:
    funnel = pd.Series(
        {"請購單": 80, "已關閉請購單": 62, "彙整列": 240, "採購單": 34, "驗收單": 41, "請款單": 22})
else:
    funnel = pd.Series({
        "請購單":       q("SELECT COUNT(*) c FROM cycle_purchase_requests")["c"].iloc[0],
        "已關閉請購單": q("SELECT COUNT(*) c FROM cycle_purchase_requests WHERE is_closed=1")["c"].iloc[0],
        "彙整列":       q("SELECT COUNT(*) c FROM cycle_purchase_summary")["c"].iloc[0],
        "採購單":       q("SELECT COUNT(*) c FROM cycle_purchase_pos")["c"].iloc[0],
        "驗收單":       q("SELECT COUNT(*) c FROM cycle_purchase_receiving")["c"].iloc[0],
        "請款單":       q("SELECT COUNT(*) c FROM cycle_purchase_payments")["c"].iloc[0],
    })

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(funnel.index, funnel.values, color=PALETTE[:len(funnel)], width=0.62)
ax.bar_label(bars, fmt="%d", padding=3, fontsize=10)
ax.set_title("各階段單據數量", fontsize=13, color=NAVY, pad=12)
ax.set_ylabel("筆數")
ax.grid(axis="y", alpha=0.25)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.margins(y=0.15)
watermark(ax)
plt.tight_layout(); plt.show()

```{admonition} 為什麼彙整列比請購單多
:class: note

彙整的粒度是「公司＋料號＋部門」，一張請購單有幾十筆明細就會攤成幾十列。
**「彙整列 ÷ 請購單」沒有業務意義**，不要拿來當轉換率。
```

## 採購單狀態與驗收完成度

`partial_received` / `received` 由 `_recompute_po_status()` 依驗收累計量回算，不是人工設的。

In [ ]:
LABELS = {"draft": "草稿", "issued": "已發出", "partial_received": "部分驗收",
          "received": "驗收完成", "cancelled": "已取消"}

if IS_DEMO:
    po_status = pd.Series({"草稿": 4, "已發出": 11, "部分驗收": 6, "驗收完成": 10, "已取消": 3})
else:
    df = q("SELECT status, COUNT(*) c FROM cycle_purchase_pos GROUP BY status")
    po_status = (df.assign(status=df["status"].map(lambda s: LABELS.get(s, s)))
                   .set_index("status")["c"]) if len(df) else pd.Series(dtype=int)

if len(po_status) == 0:
    print("目前沒有採購單。")
else:
    fig, ax = plt.subplots(figsize=(8, 3.8))
    bars = ax.bar(po_status.index, po_status.values, color=PALETTE[:len(po_status)], width=0.6)
    ax.bar_label(bars, fmt="%d", padding=3, fontsize=10)
    ax.set_title("採購單狀態分佈", fontsize=13, color=NAVY, pad=12)
    ax.set_ylabel("張數"); ax.grid(axis="y", alpha=0.25)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.margins(y=0.15)
    watermark(ax)
    plt.tight_layout(); plt.show()

## 異常稽核事件

四個「有欄位沒觸發點」的事件類型（`backfill` / `overdue` / `shortage` / `substitute`）
在這裡一定是 0——這是**已知現況**，不是資料掉了。

In [ ]:
EVENT_LABELS = {
    "receiving_variance": "驗收差異", "payment_variance": "請款差異",
    "unsummarize": "退回請購單", "revert_to_summary": "採購單退回彙整",
    "ragic_push": "拋轉 Ragic", "ragic_push_cancel": "取消拋轉",
    "backfill": "補填（無觸發點）", "overdue": "逾期（無觸發點）",
    "shortage": "缺貨（無觸發點）", "substitute": "替代品（無觸發點）",
}

if IS_DEMO:
    ev = pd.Series({"驗收差異": 9, "請款差異": 3, "退回請購單": 5,
                    "採購單退回彙整": 2, "拋轉 Ragic": 7, "取消拋轉": 2})
else:
    df = q("SELECT event_type, COUNT(*) c FROM cycle_purchase_audit_logs GROUP BY event_type ORDER BY c DESC")
    ev = (df.assign(event_type=df["event_type"].map(lambda s: EVENT_LABELS.get(s, s)))
            .set_index("event_type")["c"]) if len(df) else pd.Series(dtype=int)

never = pd.Series({EVENT_LABELS[k]: 0 for k in ("backfill", "overdue", "shortage", "substitute")})
ev = pd.concat([ev, never[~never.index.isin(ev.index)]])

fig, ax = plt.subplots(figsize=(8.4, max(3.2, 0.42 * len(ev))))
colors = ["#bdc3c7" if "無觸發點" in i else NAVY for i in ev.index]
ax.barh(ev.index[::-1], ev.values[::-1], color=colors[::-1], height=0.62)
ax.set_title("稽核事件類型分佈（灰色＝目前無觸發點）", fontsize=13, color=NAVY, pad=12)
ax.set_xlabel("筆數"); ax.grid(axis="x", alpha=0.25)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
watermark(ax)
plt.tight_layout(); plt.show()

## 口徑備忘

寫任何週採報表前先確認這幾條，可以省掉大部分「數字對不起來」的來回。

| 口徑 | 規則 |
|------|------|
| 請購金額 | 用 `cycle_purchase_requests.total_amount`（系統維護），不要自己掃明細重算 |
| 彙整數量 | 轉採購取的是 **`adjusted_qty`（調整量）**，不是 `demand_qty` |
| 進貨報表 | **只統計已送出**的驗收單（`completed` / `discrepancy`），草稿不算 |
| 日期區間快捷 | 基準日（`anchor`）是**資料最後一天**，不是今天（CLAUDE.md §8） |
| 請購單狀態 | 看 `is_closed` / `is_summarized`，**不要看 `status` 欄位**（改版殘留） |
| 人工關 vs 系統關 | 看 `close_batch_no` 前綴，`CPAUTO-` ＝ 系統關 |
| 分攤金額 | `suggested_amount`（系統建議）與 `allocated_amount`（財務實際）**並存**，報表要講清楚用哪個 |
| 彙整列 ÷ 請購單 | 分母不同（列 vs 張），**沒有業務意義** |

```{admonition} 跟 Dashboard 的數字對不起來？
:class: note

週採的 Dashboard 待辦（`GET /requests/todos`）算的是**張數**，本頁的金額圖算的是**金額**。
兩者本來就不可比。比照 Portal 既有裁示：口徑不同是正常的，不需要統一。
```